# Synthetic Organizational Document Generation for Brazilian ESG Companies

This notebook implements a synthetic data generation pipeline combining concepts from:

- Survey Paper: "On LLMs-Driven Synthetic Data Generation, Curation, and Evaluation" (Lin Long et al.)
- DocGenie Paper: "DocGenie: A Framework for High-Fidelity Synthetic Document Generation" (Harikrishnan P M et al.)

## 1. Imports and Setup

In [12]:
import os
import json
import requests
import random
from typing import List, Dict, Any
from datetime import datetime

from dotenv import load_dotenv
load_dotenv()

# Set random seeds for reproducibility
random.seed(42)

## 2. Model loading
We'll use the GAIA model (Gemma-3-Gaia-PT-BR-4b), a Brazilian Portuguese Multimodal LLM, via Ollama, which runs on a remote server.

In [16]:
class OllamaClient:
    """
    Client wrapper for Ollama server.
    """

    def __init__(self, server_url=None, model_name="brunoconterato/Gemma-3-Gaia-PT-BR-4b-it:f16"):
        """
        Initialize Ollama client.
        
        Args:
            server_url: Desktop IP (e.g., "http://192.168.1.100:11434")
                       If None, uses OLLAMA_SERVER env variable or localhost
            model_name: Ollama model to use
        """
        # Get server URL from env or parameter
        self.server_url = (
            server_url 
            or os.getenv("OLLAMA_SERVER", "http://localhost:11434")
        )
        if not self.server_url.startswith("http"):
            self.server_url = f"http://{self.server_url}"


        self.model_name = model_name
        self.api_url = f"{self.server_url}/api/generate"

    def generate(self, prompt, max_tokens=1024, temperature=0.7, stream=False):
        """
        Generate text using Ollama server.
        
        Args:
            prompt: Input text prompt
            max_tokens: Maximum tokens to generate
            temperature: Sampling temperature (0.0-1.0)
            stream: Whether to stream response
        
        Returns:
            Generated text string
        """
        payload = {
            "model": self.model_name,
            "prompt": prompt,
            "stream": stream,
            "options": {
                "num_predict": max_tokens,
                "temperature": temperature,
                "top_p": 0.9
            }
        }
        
        try:
            response = requests.post(
                self.api_url, 
                json=payload,
                timeout=120  # 2 minute timeout
            )
            
            if response.status_code == 200:
                return response.json()["response"]
            else:
                raise Exception(
                    f"Generation failed: {response.status_code} - {response.text}"
                )

        except requests.exceptions.Timeout:
            print("Generation timed out. Try reducing max_tokens.")
            raise
        except requests.exceptions.RequestException as e:
            print(f"Request failed: {e}")
            raise

def load_gaia_model():
    """
    Initialize the GAIA model client via Ollama.
    
    Returns:
        OllamaClient: Client ready for inference
    """
    print("Initializing GAIA model client...")
    
    # The client will use OLLAMA_SERVER env variable if set
    # Otherwise defaults to localhost
    client = OllamaClient()
    
    # Test connection
    try:
        test_response = client.generate("Test", max_tokens=10, temperature=0.1)
        print(f"✓ Connection successful! Server: {client.server_url}")
        print(f"✓ Model: {client.model_name}")
        return client
    except Exception as e:
        print(f"✗ Connection failed: {e}")
        print(f"  Make sure Ollama is running at {client.server_url}")
        print(f"  Set OLLAMA_SERVER environment variable if using remote server")
        raise

# Load the model
model = load_gaia_model()

Initializing GAIA model client...
✓ Connection successful! Server: http://192.168.18.9:11434
✓ Model: brunoconterato/Gemma-3-Gaia-PT-BR-4b-it:f16


## 3. Prompt Engineering
Based on the survey paper (Long et al., 2024), effective prompts contain:

- Task specification: Clear description of the generation objective
- Generation conditions: Attributes that control output diversity
- In-context demonstrations: Examples to guide the model

The DocGenie paper (Harikrishnan et al., 2025) implements this through seed-guided generation, where real document samples guide the synthesis process.

In [14]:
def create_seed_guided_prompt(
    document_type: str,
    seed_examples: List[str],
    num_samples: int = 3,
    attributes: Dict[str, Any] = None
) -> str:
    """
    Create a seed-guided generation prompt following DocGenie's approach.
    
    Args:
        document_type: Type of document to generate (e.g., "receipt", "invoice")
        seed_examples: List of example documents as text descriptions
        num_samples: Number of synthetic samples to generate
        attributes: Optional conditional attributes (domain, language, etc.)
    
    Returns:
        str: Formatted prompt for the LLM
    
    References:
        - Seed-guided generation: DocGenie paper (Harikrishnan et al., 2025)
        - Conditional prompting: Survey paper (Long et al., 2024)
    """
    
    # Task specification (from survey paper)
    task_spec = f"""Você é uma IA especializada em gerar documentos sintéticos de {document_type}.
Seu objetivo é criar documentos realistas e diversos que mantenham o estilo e estrutura 
de {document_type}s do mundo real."""
    
    # Generation conditions (conditional prompting from survey paper)
    conditions = ""
    if attributes:
        conditions = "\n\nRequisitos de Geração:"
        for key, value in attributes.items():
            conditions += f"\n- {key}: {value}"
    
    # In-context demonstrations (seed-guided approach from DocGenie)
    demonstrations = "\n\nExemplos de Referência (Documentos Semente):"
    for i, example in enumerate(seed_examples, 1):
        demonstrations += f"\n\n=== Exemplo {i} ===\n{example}"
    
    # Generation instruction
    instruction = f"""\n\nGere {num_samples} documentos de {document_type} diversos que:
1. Sigam o estilo e estrutura dos exemplos de semente
2. Contenham conteúdo realista e variado (NÃO copie os exemplos)
3. Sejam culturalmente e linguisticamente apropriados para o Brasil
4. Incluam todos os campos obrigatórios de um {document_type}

Formate cada documento claramente com um separador "---" entre eles."""
    
    return task_spec + conditions + demonstrations + instruction

## 4. Synthetic Data Generation
The survey paper describes multi-step generation strategies. A simplified version was implemented here.

In [15]:
def generate_synthetic_data(
    model_client: OllamaClient,
    prompt: str,
    max_new_tokens: int = 1024,
    temperature: float = 0.7
) -> str:
    """
    Generate synthetic documents using the LLM via Ollama.
    
    Args:
        model_client: OllamaClient instance
        prompt: Generation prompt
        max_new_tokens: Maximum tokens to generate
        temperature: Sampling temperature for diversity
    
    Returns:
        str: Generated text containing synthetic documents
    
    References:
        - Generation strategy based on survey paper (Long et al., 2024)
    """
    
    print("Generating synthetic documents...")
    print(f"  Max tokens: {max_new_tokens}")
    print(f"  Temperature: {temperature}")
    
    # Generate using Ollama
    generated_text = model_client.generate(
        prompt=prompt,
        max_tokens=max_new_tokens,
        temperature=temperature
    )
    
    print(f"✓ Generated {len(generated_text)} characters")
    
    return generated_text


def parse_generated_documents(generated_text: str) -> List[Dict[str, str]]:
    """
    Parse the generated text into individual documents.
    
    Args:
        generated_text: Raw output from the LLM
    
    Returns:
        List of documents as dictionaries
    """
    
    documents = []
    
    # Try to split by common separators
    for separator in ["---", "\n\n---\n\n", "===", "Documento "]:
        if separator in generated_text:
            parts = generated_text.split(separator)
            for i, part in enumerate(parts):
                if part.strip() and len(part.strip()) > 50:  # Minimum content length
                    documents.append({
                        "id": f"doc_{i+1}",
                        "content": part.strip(),
                        "timestamp": datetime.now().isoformat()
                    })
            break
    
    # If no separator found, treat as single document
    if not documents:
        documents.append({
            "id": "doc_1",
            "content": generated_text.strip(),
            "timestamp": datetime.now().isoformat()
        })
    
    print(f"✓ Parsed {len(documents)} documents")
    return documents